# Segmentation-Guided ResNet-50 — Google Colab

Notebook ini menjalankan `003_classification/segmentation_guided_cv_resnet50/train.py` menggunakan GPU Colab. Dataset dibaca dari satu arsip `dataset_gcolab.tar.gz` di Google Drive, tetapi hanya tiga bagian yang diekstrak ke disk lokal Colab: metadata CV, `ct_windowed`, dan probability map U-Net yang dipilih. Output training ditulis langsung ke Google Drive agar tetap tersedia ketika runtime berakhir.

Sebelum membuka Colab, buat arsip dari root repository:

```bash
tar -czf dataset_gcolab.tar.gz dataset_gcolab/
sha256sum dataset_gcolab.tar.gz > dataset_gcolab.tar.gz.sha256
```

Upload kedua file tersebut ke folder Google Drive yang dikonfigurasi pada bagian 3. Pastikan perubahan source code juga sudah di-push ke branch GitHub yang dipilih.

> Training menjalankan lima fold dan dapat memerlukan waktu lebih panjang daripada masa hidup satu runtime Colab. Script saat ini menyimpan checkpoint setiap epoch, tetapi belum memiliki CLI untuk melanjutkan run CV yang terputus. Gunakan runtime berdurasi cukup panjang dan jangan menutup sesi selama training.

## 1. Aktifkan dan periksa GPU

Pilih **Runtime > Change runtime type > GPU** sebelum menjalankan cell ini.

In [ ]:
import shutil
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU tidak tersedia. Aktifkan GPU melalui Runtime > Change runtime type."
    )

disk = shutil.disk_usage("/content")
print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")
print(f"Disk free: {disk.free / 2**30:.1f} GiB")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Konfigurasi

Sesuaikan lokasi arsip di Google Drive. `PROBABILITY_RUN_ID` harus menunjuk run yang memiliki `inference/probability_npy` di dalam arsip. Turunkan `BATCH_SIZE` jika terjadi CUDA out-of-memory.

In [ ]:
from pathlib import Path

# Source code
REPOSITORY_URL = "https://github.com/FillipusAditya/mask-guided-lung-nodule-xai.git"
REPOSITORY_BRANCH = "refactor/002-segmentation"
PROJECT_ROOT = Path("/content/mask-guided-lung-nodule-xai")

# Arsip yang sudah di-upload ke Google Drive
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/mask-guided-lung-nodule-xai")
DRIVE_DATASET_ARCHIVE = DRIVE_PROJECT_DIR / "dataset_gcolab.tar.gz"
DRIVE_DATASET_CHECKSUM = DRIVE_PROJECT_DIR / "dataset_gcolab.tar.gz.sha256"

# Lokasi hasil ekstraksi pada disk lokal Colab
LOCAL_DATA_ROOT = Path("/content/dataset_gcolab")
PROBABILITY_RUN_ID = "dc730a13-5813-4d87-b15c-3b630deb32b5"

# Output checkpoint dan metrik yang persisten
DRIVE_OUTPUT_ROOT = DRIVE_PROJECT_DIR / "classification_results"

# Training override untuk Colab
BATCH_SIZE = 32       # turunkan ke 16, 8, atau 4 jika OOM
NUM_WORKERS = 2
NUM_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 20

# Set True hanya jika ingin mengekstrak ulang data pada runtime yang sama
FORCE_REEXTRACT = False

print(f"Archive    : {DRIVE_DATASET_ARCHIVE}")
print(f"Probability: {PROBABILITY_RUN_ID}")
print(f"Output     : {DRIVE_OUTPUT_ROOT}")

## 4. Clone atau perbarui repository

Branch GitHub harus sudah memuat direktori `003_classification/segmentation_guided_cv_resnet50`.

In [ ]:
import subprocess

if (PROJECT_ROOT / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "fetch", "origin"],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "checkout", REPOSITORY_BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only", "origin", REPOSITORY_BRANCH],
        check=True,
    )
else:
    subprocess.run(
        [
            "git", "clone", "--branch", REPOSITORY_BRANCH,
            "--single-branch", REPOSITORY_URL, str(PROJECT_ROOT),
        ],
        check=True,
    )

required_script = (
    PROJECT_ROOT
    / "003_classification/segmentation_guided_cv_resnet50/train.py"
)
if not required_script.is_file():
    raise FileNotFoundError(
        f"Training script tidak ditemukan: {required_script}. "
        "Push perubahan lokal ke branch GitHub terlebih dahulu."
    )
print(f"Repository siap: {PROJECT_ROOT}")

## 5. Instal dependensi

Torch dan Torchvision bawaan Colab dipertahankan agar cocok dengan CUDA runtime. Zennit diperlukan ketika menjalankan LRP pada `test.py`.

In [ ]:
import importlib.util
from importlib.metadata import PackageNotFoundError, version
import sys

required_versions = {
    "albumentations": "2.0.8",
    "zennit": "0.5.1",
}
packages = []
for package, required_version in required_versions.items():
    try:
        installed_version = version(package)
    except PackageNotFoundError:
        installed_version = None
    if installed_version != required_version:
        packages.append(f"{package}=={required_version}")

if packages:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *packages],
        check=True,
    )

required_modules = (
    "torch", "torchvision", "albumentations", "cv2", "numpy",
    "pandas", "sklearn", "matplotlib", "tqdm",
)
missing = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f"Dependensi belum tersedia: {missing}")
print("Semua dependensi siap.")

## 6. Validasi dan ekstrak `dataset_gcolab.tar.gz`

Arsip lengkap berisi banyak data yang tidak dipakai klasifikasi. Cell ini hanya mengekstrak:

- `004_classification_cv_5fold_seed42.csv`
- `ct_windowed/`
- probability map dari `PROBABILITY_RUN_ID`

Arsip dibaca langsung dari Drive agar tidak memerlukan salinan arsip tambahan pada disk lokal.

In [ ]:
import hashlib

if not DRIVE_DATASET_ARCHIVE.is_file():
    raise FileNotFoundError(f"Arsip tidak ditemukan: {DRIVE_DATASET_ARCHIVE}")

print(f"Ukuran arsip: {DRIVE_DATASET_ARCHIVE.stat().st_size / 2**30:.2f} GiB")

# Checksum bersifat opsional tetapi direkomendasikan untuk upload besar.
if DRIVE_DATASET_CHECKSUM.is_file():
    expected_hash = DRIVE_DATASET_CHECKSUM.read_text().split()[0].strip().lower()
    digest = hashlib.sha256()
    with DRIVE_DATASET_ARCHIVE.open("rb") as archive_file:
        for chunk in iter(lambda: archive_file.read(16 * 1024 * 1024), b""):
            digest.update(chunk)
    actual_hash = digest.hexdigest()
    if actual_hash != expected_hash:
        raise RuntimeError("Checksum arsip tidak cocok; upload kemungkinan rusak.")
    print("Checksum SHA-256 valid.")
else:
    print("Checksum tidak ditemukan; validasi SHA-256 dilewati.")

metadata_member = (
    "dataset_gcolab/000_dataset/_segmentation_dataset_v2/"
    "004_classification_cv_5fold_seed42.csv"
)
ct_member = (
    "dataset_gcolab/000_dataset/_segmentation_dataset_v2/ct_windowed"
)
probability_member = (
    "dataset_gcolab/segmentation_results/unet_holdout_split/"
    f"{PROBABILITY_RUN_ID}/inference/probability_npy"
)
required_members = (metadata_member, ct_member, probability_member)
extract_marker = LOCAL_DATA_ROOT / f".classification_extract_{PROBABILITY_RUN_ID}"

if FORCE_REEXTRACT or not extract_marker.is_file():
    print("Mengekstrak data training yang diperlukan...", flush=True)
    subprocess.run(
        [
            "tar", "--checkpoint=5000",
            "--checkpoint-action=echo=Extract checkpoint %u",
            "-xzf", str(DRIVE_DATASET_ARCHIVE),
            "-C", "/content", *required_members,
        ],
        check=True,
    )
    extract_marker.touch()
    print("Ekstraksi selesai.")
else:
    print("Data lokal sudah diekstrak; menggunakan hasil yang tersedia.")

## 7. Hubungkan data lokal dan output Google Drive

Symlink membuat struktur path yang diharapkan `train.py` tanpa menggandakan data. `classification_results` diarahkan ke Drive sehingga best model, checkpoint, log, dan metrik bersifat persisten.

In [ ]:
import os

local_dataset_dir = LOCAL_DATA_ROOT / "000_dataset"
local_segmentation_dir = LOCAL_DATA_ROOT / "segmentation_results"

for required_dir in (local_dataset_dir, local_segmentation_dir):
    if not required_dir.is_dir():
        raise FileNotFoundError(f"Direktori hasil ekstraksi tidak ada: {required_dir}")

DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
links = {
    PROJECT_ROOT / "000_dataset": local_dataset_dir,
    PROJECT_ROOT / "segmentation_results": local_segmentation_dir,
    PROJECT_ROOT / "classification_results": DRIVE_OUTPUT_ROOT,
}

for link, target in links.items():
    if link.is_symlink():
        if link.resolve() != target.resolve():
            raise RuntimeError(f"Symlink menunjuk target berbeda: {link}")
    elif link.exists():
        raise FileExistsError(
            f"Path sudah ada dan bukan symlink: {link}. Gunakan clone Colab yang bersih."
        )
    else:
        os.symlink(target, link, target_is_directory=True)
    print(f"{link} -> {target}")

## 8. Preflight dataset dan model

Cell ini memeriksa semua path dalam metadata, memastikan jumlah probability map lengkap, lalu menjalankan forward satu sampel sebelum training panjang dimulai.

In [ ]:
import importlib
import pandas as pd

dataset_root = local_dataset_dir / "_segmentation_dataset_v2"
metadata_path = dataset_root / "004_classification_cv_5fold_seed42.csv"
probability_root = (
    local_segmentation_dir / "unet_holdout_split" / PROBABILITY_RUN_ID
    / "inference/probability_npy"
)

metadata = pd.read_csv(metadata_path)
missing_ct = [
    path for path in metadata["ct_windowed_path"]
    if not (dataset_root / str(path)).is_file()
]
missing_probability = [
    name for name in metadata["filename"]
    if not (probability_root / Path(str(name)).name).is_file()
]
if missing_ct or missing_probability:
    raise FileNotFoundError(
        f"Missing CT={len(missing_ct)}, probability={len(missing_probability)}"
    )

dataset_module = importlib.import_module(
    "003_classification.segmentation_guided_cv_resnet50.dataset"
)
transform_module = importlib.import_module(
    "003_classification.segmentation_guided_cv_resnet50.transforms"
)
model_module = importlib.import_module(
    "003_classification.segmentation_guided_cv_resnet50.model"
)

validation_dataset = dataset_module.ProbabilityGuidedClassificationDataset(
    root_dir=dataset_root,
    metadata_path=metadata_path,
    split="val",
    cv_fold=0,
    probability_root=probability_root,
    transform=transform_module.build_val_transform(
        224, 224, (0.485, 0.456, 0.406), (0.229, 0.224, 0.225), 42
    ),
)
sample, target = validation_dataset[0]
smoke_model = model_module.SegmentationGuidedResNet50(
    num_classes=2, weights=None
).to("cuda").eval()
with torch.no_grad():
    smoke_output = smoke_model(sample.unsqueeze(0).to("cuda"))
del smoke_model
torch.cuda.empty_cache()

print(f"Metadata rows       : {len(metadata):,}")
print(f"Validation fold 0   : {len(validation_dataset):,}")
print(f"Input sample        : {tuple(sample.shape)}")
print(f"Target              : {int(target)}")
print(f"Model output        : {tuple(smoke_output.shape)}")
print(f"Probability maps    : {len(list(probability_root.glob('*.npy'))):,}")
print("Preflight berhasil.")

## 9. Jalankan training 5-fold

Training membuat run baru. Jangan menjalankan ulang cell yang sama setelah run mulai kecuali Anda memang ingin membuat run baru. Jika OOM terjadi, turunkan `BATCH_SIZE` pada konfigurasi lalu restart runtime agar VRAM benar-benar bersih.

In [ ]:
importlib.invalidate_caches()
train_module = importlib.import_module(
    "003_classification.segmentation_guided_cv_resnet50.train"
)

# Override eksplisit untuk runtime Colab dan data hasil ekstraksi.
train_module.DATASET_ROOT = dataset_root
train_module.METADATA_PATH = metadata_path
train_module.PROBABILITY_ROOT = probability_root
train_module.BATCH_SIZE = BATCH_SIZE
train_module.NUM_WORKERS = NUM_WORKERS
train_module.PERSISTENT_WORKERS = NUM_WORKERS > 0
train_module.NUM_EPOCHS = NUM_EPOCHS
train_module.EARLY_STOPPING_PATIENCE = EARLY_STOPPING_PATIENCE
train_module.PIN_MEMORY = True
train_module.DEVICE = torch.device("cuda")

print(f"Output run : {train_module.OUTPUT_DIR}")
print(f"Batch size : {train_module.BATCH_SIZE}")
print(f"Epoch/fold : {train_module.NUM_EPOCHS}")
print(f"Patience   : {train_module.EARLY_STOPPING_PATIENCE}")
train_module.main()

## 10. Periksa output training

In [ ]:
from IPython.display import display

results_root = DRIVE_OUTPUT_ROOT / "segmentation_guided_cv_resnet50"
runs = sorted(
    [path for path in results_root.glob("cv_result_*") if path.is_dir()],
    key=lambda path: path.stat().st_mtime,
)
if not runs:
    raise FileNotFoundError(f"Belum ada output training di {results_root}")
latest_run = runs[-1]
print(f"Latest run: {latest_run}")

summary_path = latest_run / "cv_summary.csv"
if summary_path.is_file():
    display(pd.read_csv(summary_path))
else:
    completed_folds = sorted(
        path.name for path in latest_run.glob("fold_*")
        if (path / "best_model.pth").is_file()
    )
    print(f"Training belum selesai. Fold dengan model: {completed_folds}")

## 11. Opsional: test, Grad-CAM, dan LRP

Jalankan hanya setelah kelima `fold_0` sampai `fold_4` memiliki `best_model.pth`. Proses LRP untuk seluruh holdout cukup berat; gunakan `MAX_TEST_SAMPLES` untuk smoke test.

In [ ]:
RUN_TEST = False
MAX_TEST_SAMPLES = 8  # gunakan None untuk seluruh holdout
TEST_BATCH_SIZE = 2

if RUN_TEST:
    command = [
        sys.executable, "-m",
        "003_classification.segmentation_guided_cv_resnet50.test",
        str(latest_run),
        "--batch-size", str(TEST_BATCH_SIZE),
        "--num-workers", str(NUM_WORKERS),
        "--device", "cuda",
    ]
    if MAX_TEST_SAMPLES is not None:
        command.extend(["--max-samples", str(MAX_TEST_SAMPLES)])
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
    print(f"Test output: {latest_run / 'test'}")
else:
    print("Test dilewati. Set RUN_TEST=True setelah training selesai.")